# LeetCode 2174 — Remove All Ones With Row and Column Flips II bằng Python

Link đề: https://leetcode.com/problems/remove-all-ones-with-row-and-column-flips-ii/ · Độ khó: **Medium** (🔒 Premium) · Chủ đề: Array, Bit Manipulation, Breadth-First Search, Matrix

> **Lưu ý Premium:** đây là bài Premium; đề bài trong notebook được viết theo bản mô tả công khai của doocs/leetcode. Nếu bạn có tài khoản Premium, hãy **đối chiếu lại với đề gốc** (tên hàm, ràng buộc, ví dụ).

Skeleton LeetCode:

```python
class Solution:
    def removeOnes(self, grid: List[List[int]]) -> int:
        ...
```

Ràng buộc then chốt: `m · n ≤ 15` ⇒ toàn bộ lưới vừa **một số nguyên 15 bit**, chỉ có `2¹⁵ = 32 768` trạng thái. Bài toán là **đường đi ngắn nhất trên đồ thị trạng thái**. Notebook đi từ **quay lui thử mọi thứ tự thao tác** → **DFS + ghi nhớ trên bitmask** → **BFS theo tầng** → **DP từ dưới lên theo thứ tự số của mask** → **tìm sâu dần (IDDFS) có cận dưới**.

## 1. Mục tiêu học

1. Mã hóa lưới nhỏ thành **bitmask**: ô `(i, j)` ↔ bit `i·n + j`.
2. Tính sẵn **mặt nạ xóa** của mỗi ô (cả hàng `i` và cột `j`) để mỗi thao tác chỉ là `mask & ~clear[c]`.
3. Giải "số thao tác ít nhất" bằng **BFS** trên đồ thị trạng thái, và bằng **DP có ghi nhớ**.
4. Nhận ra vì sao **tham lam** (chọn ô xóa nhiều số 1 nhất) và các công thức "đếm hàng/cột" **sai**, bằng phản ví dụ chạy thật.
5. Hiểu điều kiện **chỉ được chọn ô đang là 1** ảnh hưởng thế nào tới không gian trạng thái.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `grid` cỡ `m × n`. Một thao tác: chọn `(i, j)` với `grid[i][j] == 1`, rồi đặt **mọi ô trong hàng `i` và cột `j`** về 0. Trả về số thao tác ít nhất để xóa hết các số 1.

**Ví dụ 1:** `[[1,1,1],[1,1,1],[0,1,0]]` → `2` (chọn `(1,1)` rồi `(0,0)`).

**Ví dụ 2:** `[[0,1,0],[1,0,1],[0,1,0]]` → `2` (chọn `(1,0)` rồi `(2,1)`; **không** được chọn `(1,1)` vì nó là 0).

**Ví dụ 3:** `[[0,0],[0,0]]` → `0`.

**Ràng buộc:** `1 ≤ m, n ≤ 15`, `1 ≤ m · n ≤ 15`, `grid[i][j] ∈ {0, 1}`.

In [1]:
import random
import time
from collections import deque
from functools import lru_cache
from typing import List

examples = [([[1, 1, 1], [1, 1, 1], [0, 1, 0]], 2), ([[0, 1, 0], [1, 0, 1], [0, 1, 0]], 2), ([[0, 0], [0, 0]], 0)]

def draw(grid, mark=None):
    for i, row in enumerate(grid):
        print("    " + " ".join("◆" if mark == (i, j) else ("█" if x else "·") for j, x in enumerate(row)))

for g, exp in examples:
    draw(g); print("    -> kỳ vọng", exp)

    █ █ █
    █ █ █
    · █ ·
    -> kỳ vọng 2
    · █ ·
    █ · █
    · █ ·
    -> kỳ vọng 2
    · ·
    · ·
    -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Lưới → bitmask

In [2]:
def to_mask(grid):
    m, n = len(grid), len(grid[0]); mask = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                mask |= 1 << (i * n + j)
    return mask

def from_mask(mask, m, n):
    return [[mask >> (i * n + j) & 1 for j in range(n)] for i in range(m)]

g = examples[0][0]
mk = to_mask(g)
print("mask =", mk, "=", format(mk, "09b"), "(bit 0 ở bên phải) | dựng lại:", from_mask(mk, 3, 3))

mask = 191 = 010111111 (bit 0 ở bên phải) | dựng lại: [[1, 1, 1], [1, 1, 1], [0, 1, 0]]


### 3.2 Mặt nạ xóa hàng + cột của một ô

In [3]:
def clear_masks(m, n):
    out = []
    for i in range(m):
        for j in range(n):
            c = 0
            for k in range(n): c |= 1 << (i * n + k)       # hàng i
            for k in range(m): c |= 1 << (k * n + j)       # cột j
            out.append(c)
    return out

cm = clear_masks(3, 3)
print("mặt nạ xóa của ô (1,1):"); draw(from_mask(cm[4], 3, 3))

mặt nạ xóa của ô (1,1):
    · █ ·
    █ █ █
    · █ ·


### 3.3 Duyệt các bit 1 của mask

`while x: low = x & -x; b = low.bit_length() − 1; x ^= low` — lấy lần lượt từng bit 1.

### 3.4 BFS trên đồ thị trạng thái

Mỗi trạng thái là một mask; cạnh = một thao tác hợp lệ. Mọi cạnh có trọng số 1 ⇒ BFS cho **số cạnh ít nhất** tới trạng thái `0`.

In [4]:
def bits(x):
    out = []
    while x:
        low = x & -x; out.append(low.bit_length() - 1); x ^= low
    return out

print("các bit 1 của 0b101101:", bits(0b101101))

các bit 1 của 0b101101: [0, 2, 3, 5]


## 4. Ý tưởng nền tảng / trực giác

Mỗi thao tác xóa trọn **một hàng và một cột**. Ví dụ 1: chọn `(1,1)` xóa hàng 1 và cột 1, còn lại `(0,0)`, `(0,2)`: hai ô cùng hàng 0 ⇒ chọn `(0,0)` xóa nốt ⇒ 2 thao tác. Một thao tác không đủ vì không có ô nào mà hàng + cột của nó phủ cả 7 số 1.

Ví dụ 2 cho thấy ràng buộc "ô phải là 1": nếu được chọn `(1,1)` thì **một** thao tác xóa cả 4 số 1 (hàng 1 + cột 1), nhưng `(1,1) = 0` ⇒ phải 2 thao tác.

Vì thao tác chỉ làm **mất** số 1 (không bao giờ thêm), trạng thái mới luôn là **tập con nghiêm ngặt** của trạng thái cũ ⇒ đồ thị trạng thái **không có chu trình** (DAG) ⇒ DP được.

In [5]:
g = examples[1][0]
print("Nếu (sai luật) chọn ô (1,1):"); draw(from_mask(to_mask(g) & ~clear_masks(3, 3)[4], 3, 3))
print("Theo luật: chọn (1,0) rồi (2,1):")
s1 = to_mask(g) & ~clear_masks(3, 3)[3]; draw(from_mask(s1, 3, 3))
s2 = s1 & ~clear_masks(3, 3)[7]; draw(from_mask(s2, 3, 3))

Nếu (sai luật) chọn ô (1,1):
    · · ·
    · · ·
    · · ·
Theo luật: chọn (1,0) rồi (2,1):
    · █ ·
    · · ·
    · █ ·
    · · ·
    · · ·
    · · ·


## 5. Cách 1 — Brute Force / baseline: quay lui thử mọi thứ tự

Thử chọn lần lượt từng ô đang là 1, đệ quy, lấy số bước nhỏ nhất. Không ghi nhớ ⇒ cùng một trạng thái bị giải lại nhiều lần. Cắt tỉa đơn giản: không đi sâu hơn `best − 1`.

In [6]:
def remove_ones_brute(grid):
    m, n = len(grid), len(grid[0])
    clear = clear_masks(m, n)
    best = [m * n]
    def go(mask, depth):
        if depth >= best[0]:
            return
        if mask == 0:
            best[0] = depth; return
        for b in bits(mask):
            go(mask & ~clear[b], depth + 1)
    go(to_mask(grid), 0)
    return best[0] if to_mask(grid) else 0

for g, exp in examples:
    print(remove_ones_brute(g), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp:** số nhánh tới `15 · 14 · …` — mũ, dù có cắt tỉa. **Nút thắt:** trạng thái lặp lại (thứ tự thao tác khác nhau dẫn tới cùng mask) ⇒ **ghi nhớ** theo mask, hoặc BFS với `visited`.

## 6. Ý tưởng thuật toán / bước đột phá

**Đồ thị trạng thái:** đỉnh = mask (≤ `2^{mn}` đỉnh), cạnh `mask → mask & ~clear[b]` với mỗi bit `b` của `mask`. Đáp án = khoảng cách ngắn nhất từ `start` tới `0`.

- **BFS**: lan theo tầng; tầng đầu tiên chứa `0` là đáp án.
- **DP**: `f(0) = 0`, `f(mask) = 1 + min_{b ∈ mask} f(mask & ~clear[b])`. Vì `mask & ~clear[b] < mask` (mất ít nhất bit `b`), duyệt `mask` tăng dần là thứ tự tô-pô hợp lệ.

**Quy tắc then chốt:** chỉ thử `b` là **bit 1** của mask hiện tại (ô đang là 1).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS + ghi nhớ (`lru_cache`)

In [7]:
def remove_ones_memo(grid):
    m, n = len(grid), len(grid[0])
    clear = clear_masks(m, n)

    @lru_cache(maxsize=None)
    def f(mask):
        if mask == 0:
            return 0
        return 1 + min(f(mask & ~clear[b]) for b in bits(mask))

    return f(to_mask(grid))

for g, exp in examples:
    print(remove_ones_memo(g), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp:** mỗi mask **đạt được** tính một lần, mỗi lần `O(mn)` ⇒ `O(R · mn)` với `R ≤ 2^{mn}` trạng thái đạt được. Độ sâu đệ quy ≤ số thao tác ≤ `min(m, n)` + … (mỗi thao tác xóa một hàng) ⇒ ≤ 15, an toàn.

### 7.2 Cách 3 — BFS theo tầng

In [8]:
def remove_ones_bfs(grid):
    m, n = len(grid), len(grid[0])
    clear = clear_masks(m, n)
    start = to_mask(grid)
    if start == 0:
        return 0
    seen = {start}
    q = deque([start])
    steps = 0
    while q:
        steps += 1
        for _ in range(len(q)):
            mask = q.popleft()
            for b in bits(mask):
                nxt = mask & ~clear[b]
                if nxt == 0:
                    return steps
                if nxt not in seen:
                    seen.add(nxt); q.append(nxt)
    return -1                                        # không xảy ra

for g, exp in examples:
    print(remove_ones_bfs(g), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(R · mn)`; thường dừng sớm vì đáp án nhỏ (≤ số hàng).

### 7.3 Cách 4 — DP từ dưới lên trên mọi mask con của `start`

Chỉ những mask là **tập con** của `start` mới có thể xuất hiện. Duyệt mọi tập con của `start` theo thứ tự **tăng** (dùng mảng hoặc dict), `f[sub] = 1 + min f[sub & ~clear[b]]`.

In [9]:
def remove_ones_bottom_up(grid):
    m, n = len(grid), len(grid[0])
    clear = clear_masks(m, n)
    start = to_mask(grid)
    subs = []
    sub = start
    while True:                                     # liệt kê mọi tập con của start
        subs.append(sub)
        if sub == 0:
            break
        sub = (sub - 1) & start
    f = {}
    for s in reversed(subs):                        # reversed -> thứ tự tăng dần
        if s == 0:
            f[s] = 0
        else:
            f[s] = 1 + min(f[s & ~clear[b]] for b in bits(s))
    return f[start]

for g, exp in examples:
    print(remove_ones_bottom_up(g), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Vì sao `s & ~clear[b]` đã có trong `f`?** Nó là tập con của `s` (nên cũng là tập con của `start`) và nhỏ hơn `s` về giá trị số ⇒ đã được tính trước trong thứ tự tăng. **Độ phức tạp:** `O(2^{popcount(start)} · mn)` — xét cả các tập con **không đạt được**, nên có thể chậm hơn BFS, nhưng không cần hàng đợi.

### 7.4 Cách 5 — Tìm sâu dần (IDDFS) với cận dưới

Thử độ sâu `d = 1, 2, …`: DFS có giới hạn độ sâu `d`, cắt nhánh bằng **cận dưới**: mỗi thao tác xóa tối đa `m + n − 1` ô (một hàng + một cột) ⇒ cần ít nhất `ceil(số_1 / (m + n − 1))` thao tác nữa. Thêm ghi nhớ "mask đã thất bại khi còn `r` bước" để không thử lại.

In [10]:
def remove_ones_iddfs(grid):
    m, n = len(grid), len(grid[0])
    clear = clear_masks(m, n)
    start = to_mask(grid)
    if start == 0:
        return 0
    cap = m + n - 1
    def dfs(mask, left, failed):
        if mask == 0:
            return True
        if left == 0 or -(-bin(mask).count("1") // cap) > left:
            return False
        if failed.get(mask, -1) >= left:
            return False
        for b in bits(mask):
            if dfs(mask & ~clear[b], left - 1, failed):
                return True
        failed[mask] = left
        return False
    d = 1
    while not dfs(start, d, {}):
        d += 1
    return d

for g, exp in examples:
    print(remove_ones_iddfs(g), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp:** xấu nhất tương đương DFS mũ, nhưng với cận dưới + ghi nhớ thất bại thì rất nhanh vì đáp án nhỏ.

## 8. TRACE từng bước

### 8.1 Trace BFS trên ví dụ 1 — in từng tầng

In [11]:
def trace_bfs(grid):
    m, n = len(grid), len(grid[0]); clear = clear_masks(m, n); start = to_mask(grid)
    q = deque([start]); seen = {start}; steps = 0
    parent = {start: None}
    while q:
        steps += 1
        layer = []
        for _ in range(len(q)):
            mask = q.popleft()
            for b in bits(mask):
                nxt = mask & ~clear[b]
                if nxt not in seen:
                    seen.add(nxt); parent[nxt] = (mask, b); q.append(nxt); layer.append(nxt)
        print(f"  tầng {steps}: {len(layer)} trạng thái mới, số bit 1 ít nhất = {min(bin(x).count('1') for x in layer) if layer else '-'}")
        if 0 in parent:
            break
    path = []; cur = 0
    while parent[cur] is not None:
        pm, b = parent[cur]; path.append((pm, b)); cur = pm
    for pm, b in reversed(path):
        i, j = divmod(b, n)
        print(f"  chọn ô ({i},{j}):"); draw(from_mask(pm, m, n), mark=(i, j))
    print("  => số thao tác:", steps)
    return steps

_ = trace_bfs(examples[0][0])

  tầng 1: 7 trạng thái mới, số bit 1 ít nhất = 2
  tầng 2: 6 trạng thái mới, số bit 1 ít nhất = 0
  chọn ô (0,0):
    ◆ █ █
    █ █ █
    · █ ·
  chọn ô (1,1):
    · · ·
    · ◆ █
    · █ ·
  => số thao tác: 2


### 8.2 Trace DP ghi nhớ trên lưới tự dựng 3×3 (đã đối chiếu vét cạn — đối chiếu lại đề gốc)

In [12]:
def trace_memo(grid):
    m, n = len(grid), len(grid[0]); clear = clear_masks(m, n); calls = []
    @lru_cache(maxsize=None)
    def f(mask):
        if mask == 0:
            return 0
        opts = [(1 + f(mask & ~clear[b]), b) for b in bits(mask)]
        v, b = min(opts)
        calls.append((bin(mask).count("1"), mask, v, divmod(b, n)))
        return v
    r = f(to_mask(grid))
    for k, mask, v, cell in sorted(calls, reverse=True)[:6]:
        print(f"  mask có {k} số 1 ({format(mask, f'0{m * n}b')}): f = {v}, ô tốt nhất {cell}")
    print(f"  ... tổng {len(calls)} trạng thái khác 0 được giải; đáp án {r}")
    return r

g8 = [[1, 1, 1], [1, 0, 1], [1, 1, 0]]
assert trace_memo(g8) == remove_ones_brute(g8)

  mask có 7 số 1 (011101111): f = 2, ô tốt nhất (0, 1)
  mask có 4 số 1 (011000011): f = 2, ô tốt nhất (0, 0)
  mask có 4 số 1 (000101101): f = 2, ô tốt nhất (0, 0)
  mask có 3 số 1 (011001000): f = 1, ô tốt nhất (2, 0)
  mask có 3 số 1 (010000110): f = 1, ô tốt nhất (0, 1)
  mask có 3 số 1 (001101000): f = 1, ô tốt nhất (1, 0)
  ... tổng 15 trạng thái khác 0 được giải; đáp án 2


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def removeOnes(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        start = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j]:
                    start |= 1 << (i * n + j)
        if start == 0:
            return 0
        clear = []
        for i in range(m):
            for j in range(n):
                c = 0
                for k in range(n):
                    c |= 1 << (i * n + k)
                for k in range(m):
                    c |= 1 << (k * n + j)
                clear.append(c)
        seen = {start}
        q = deque([start])
        steps = 0
        while q:
            steps += 1
            for _ in range(len(q)):
                mask = q.popleft()
                x = mask
                while x:                                  # chỉ chọn ô đang là 1
                    low = x & -x
                    x ^= low
                    nxt = mask & ~clear[low.bit_length() - 1]
                    if nxt == 0:
                        return steps
                    if nxt not in seen:
                        seen.add(nxt)
                        q.append(nxt)
        return -1

for g, exp in examples:
    assert Solution().removeOnes(g) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Mô hình đúng.** Trạng thái lưới hoàn toàn xác định bởi tập ô đang là 1 (mask). Thao tác trên ô `b` (phải là bit 1) cho `mask & ~clear[b]` — đúng định nghĩa "đặt hàng và cột về 0".
2. **BFS.** Mọi cạnh trọng số 1; BFS thăm các trạng thái theo khoảng cách tăng dần, `seen` đảm bảo mỗi trạng thái vào hàng đợi một lần ở khoảng cách nhỏ nhất. Lần đầu sinh ra `0` ở tầng `steps` ⇒ `steps` là số thao tác nhỏ nhất.
3. **DP.** Đồ thị không chu trình (mỗi cạnh làm mask giảm nghiêm ngặt). Quy nạp theo giá trị mask: `f(mask)` = 1 + min trên các bước đầu tiên khả dĩ — đúng định nghĩa đường đi ngắn nhất trên DAG.
4. **Luôn kết thúc.** Mỗi thao tác xóa ít nhất ô được chọn ⇒ sau ≤ `popcount(start)` thao tác về 0; BFS chắc chắn tìm thấy.

## 11. Phân tích Time / Space Complexity

Gọi `K = m · n ≤ 15`, `R` = số trạng thái đạt được (`≤ 2^K = 32 768`).

| Cách | Time | Space |
|---|---|---|
| 1. Quay lui | mũ (`O(K!)` xấu nhất) | `O(K)` |
| 2. DFS + ghi nhớ | `O(R · K)` | `O(R)` |
| 3. BFS | `O(R · K)`, dừng sớm | `O(R)` |
| 4. DP mọi tập con | `O(2^{popcount} · K)` | `O(2^{popcount})` |
| 5. IDDFS + cận | mũ xấu nhất, thực tế nhanh | `O(R)` (ghi nhớ thất bại) |

In [14]:
rnd = random.Random(2174)
shapes = [(3, 5), (5, 3), (1, 15), (15, 1), (2, 7)]
grids = [[[1 if rnd.random() < 0.7 else 0 for _ in range(n)] for _ in range(m)] for m, n in shapes] + [[[1] * 5 for _ in range(3)]]
for name, fn in [("memo", remove_ones_memo), ("bfs", remove_ones_bfs), ("bottom_up", remove_ones_bottom_up),
                 ("iddfs", remove_ones_iddfs), ("Solution", Solution().removeOnes)]:
    t0 = time.perf_counter(); res = [fn(g) for g in grids]; dt = time.perf_counter() - t0
    print(f"{name:<10}: {res}  {dt * 1000:8.1f} ms")

memo      : [2, 3, 1, 1, 2, 3]       0.2 ms
bfs       : [2, 3, 1, 1, 2, 3]       0.1 ms
bottom_up : [2, 3, 1, 1, 2, 3]      58.4 ms
iddfs     : [2, 3, 1, 1, 2, 3]       0.1 ms
Solution  : [2, 3, 1, 1, 2, 3]       0.1 ms


## 12. Bộ test

In [15]:
ALL = {
    "memo": remove_ones_memo,
    "bfs": remove_ones_bfs,
    "bottom_up": remove_ones_bottom_up,
    "iddfs": remove_ones_iddfs,
    "Solution": lambda g: Solution().removeOnes(g),
}

def check(grid, exp=None, brute=True):
    ref = remove_ones_brute(grid) if brute else remove_ones_bfs(grid)
    for name, fn in ALL.items():
        got = fn([row[:] for row in grid])
        assert got == ref, (name, grid, got, ref)
    if exp is not None:
        assert ref == exp, (grid, ref, exp)
    return ref

for g, exp in examples:
    check(g, exp)
print("Ví dụ đề: OK")

check([[1]], 1)
check([[0]], 0)
check([[1] * 15], 1)                       # một hàng: một thao tác
check([[1]] * 15, 1)                       # một cột
check([[1, 0, 0], [0, 1, 0], [0, 0, 1]], 3)   # đường chéo: mỗi thao tác chỉ xóa được đúng một số 1
check([[0, 1], [1, 1]], 1)
check([[1, 1, 1], [1, 0, 1], [1, 1, 0]], 2)
print("Edge case (ví dụ tự dựng, kỳ vọng đã đối chiếu vét cạn): OK")

rnd = random.Random(2021)
for _ in range(1500):
    m = rnd.randint(1, 4); n = rnd.randint(1, min(4, 15 // m))
    p = rnd.random()
    check([[1 if rnd.random() < p else 0 for _ in range(n)] for _ in range(m)])
print("1500 lưới ngẫu nhiên nhỏ (≤ 4×4… với m·n ≤ 15) so với quay lui: OK")

for _ in range(60):
    m = rnd.choice([1, 2, 3, 5, 15]); n = 15 // m
    check([[rnd.randint(0, 1) for _ in range(n)] for _ in range(m)], brute=False)
print("60 lưới m·n = 15 (đối chiếu chéo): OK")

Ví dụ đề: OK


Edge case (ví dụ tự dựng, kỳ vọng đã đối chiếu vét cạn): OK

1500 lưới ngẫu nhiên nhỏ (≤ 4×4… với m·n ≤ 15) so với quay lui: OK


60 lưới m·n = 15 (đối chiếu chéo): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: mỗi bước chọn ô xóa được **nhiều số 1 nhất**

In [16]:
def bug_greedy(grid):
    g = [r[:] for r in grid]; m, n = len(g), len(g[0]); ops = 0
    while any(map(any, g)):
        best = None
        for i in range(m):
            for j in range(n):
                if g[i][j]:
                    c = sum(g[i]) + sum(g[k][j] for k in range(m)) - 1
                    if best is None or c > best[0]:
                        best = (c, i, j)
        _, i, j = best
        for k in range(n): g[i][k] = 0
        for k in range(m): g[k][j] = 0
        ops += 1
    return ops

g = [[1, 1, 1], [1, 0, 1], [1, 1, 0]]
print("tham lam (sai):", bug_greedy(g), "| đúng:", Solution().removeOnes(g))

tham lam (sai): 3 | đúng: 2


**Vì sao sai:** tham lam chọn `(0,0)` (xóa 5 số 1 — nhiều nhất) nhưng để lại `(1,2)` và `(2,1)` nằm ở hàng khác **và** cột khác nhau ⇒ cần thêm 2 thao tác, tổng 3. Lời giải tối ưu chỉ cần 2 (DP ở mục 8.2 in ra ô tốt nhất cho từng trạng thái). **Sửa:** BFS/DP xét mọi lựa chọn.

### Lỗi 2 — Cho phép chọn ô đang là 0

In [17]:
def bug_any_cell(grid):
    m, n = len(grid), len(grid[0]); clear = clear_masks(m, n)
    @lru_cache(maxsize=None)
    def f(mask):
        if mask == 0: return 0
        return 1 + min(f(mask & ~clear[b]) for b in range(m * n) if mask & clear[b])   # SAI: mọi ô, kể cả ô 0
    return f(to_mask(grid))

g = examples[1][0]
print("cho chọn ô 0 (sai):", bug_any_cell(g), "| đúng:", Solution().removeOnes(g))

cho chọn ô 0 (sai): 1 | đúng: 2


**Sửa:** chỉ duyệt các bit 1 của mask hiện tại.

### Lỗi 3 — Đánh số bit sai: `i * m + j` thay vì `i * n + j`

In [18]:
def bug_index(grid):
    m, n = len(grid), len(grid[0])
    start = sum(1 << (i * m + j) for i in range(m) for j in range(n) if grid[i][j])       # SAI
    clear = []
    for i in range(m):
        for j in range(n):
            c = 0
            for k in range(n): c |= 1 << (i * m + k)
            for k in range(m): c |= 1 << (k * m + j)
            clear.append(c)
    bit_of = {i * m + j: idx for idx, (i, j) in enumerate((i, j) for i in range(m) for j in range(n))}
    @lru_cache(maxsize=None)
    def f(mask):
        if mask == 0: return 0
        return 1 + min(f(mask & ~clear[bit_of[b]]) for b in bits(mask) if b in bit_of)
    return f(start)

g = [[1, 0, 1], [0, 1, 0]]          # 2 x 3 (không vuông)
print("đánh số sai:", bug_index(g), "| đúng:", Solution().removeOnes(g))

đánh số sai: 1 | đúng: 2


**Vì sao sai:** với lưới không vuông, `i·m + j` làm hai ô khác nhau **trùng bit** (ở đây `(0,2)` và `(1,0)` cùng bit 2) ⇒ trạng thái sai. Lưới vuông che giấu lỗi này — luôn test lưới `m ≠ n`. **Sửa:** `i·n + j` (`n` = số cột).

### Lỗi 4 — "Đáp án = min(số hàng có 1, số cột có 1)"

In [19]:
def bug_min_rows_cols(grid):
    r = sum(1 for row in grid if any(row))
    c = sum(1 for j in range(len(grid[0])) if any(row[j] for row in grid))
    return min(r, c)

g = [[0, 1], [1, 1]]
print("min(hàng, cột) (sai):", bug_min_rows_cols(g), "| đúng:", Solution().removeOnes(g))
g = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
print("trên đường chéo:", bug_min_rows_cols(g), "| đúng:", Solution().removeOnes(g), "(trùng hợp đúng)")

min(hàng, cột) (sai): 2 | đúng: 1
trên đường chéo: 3 | đúng: 3 (trùng hợp đúng)


**Vì sao sai:** một thao tác xóa đồng thời **một hàng và một cột**, nên có thể "ăn" cả hai chiều cùng lúc. Đây chỉ là **cận trên** khi mọi hàng có ô chọn được — không phải đáp án.

### Lỗi 5 — BFS không dùng `seen` ⇒ bùng nổ hàng đợi (đếm số trạng thái đẩy vào)

In [20]:
def bfs_pushes(grid, use_seen):
    m, n = len(grid), len(grid[0]); clear = clear_masks(m, n); start = to_mask(grid)
    q = deque([start]); seen = {start}; pushes = 0; steps = 0
    while q:
        steps += 1
        for _ in range(len(q)):
            mask = q.popleft()
            for b in bits(mask):
                nxt = mask & ~clear[b]
                if nxt == 0:
                    return steps, pushes
                if not use_seen or nxt not in seen:
                    seen.add(nxt); q.append(nxt); pushes += 1
    return -1, pushes

g = [[1, 1, 1, 1, 1], [1, 0, 1, 0, 1], [1, 1, 0, 1, 1]]
print("không seen:", bfs_pushes(g, False), "| có seen:", bfs_pushes(g, True), "  (bước, số lần đẩy vào hàng đợi)")

không seen: (3, 88) | có seen: (3, 37)   (bước, số lần đẩy vào hàng đợi)


**Vì sao là lỗi:** kết quả vẫn đúng nhưng số trạng thái đẩy vào hàng đợi tăng vọt (cùng một mask được đẩy nhiều lần theo các thứ tự thao tác khác nhau) — với lưới lớn hơn sẽ TLE/MLE. **Sửa:** `seen` (hoặc DP ghi nhớ).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Ràng buộc kiểu `m · n ≤ 15/16/20` ⇒ **bitmask toàn bộ lưới**.
- "Số thao tác ít nhất" + trạng thái rời rạc nhỏ ⇒ **BFS**; trạng thái chỉ giảm ⇒ cũng có thể **DP trên DAG**.
- Tính trước mặt nạ của từng thao tác ⇒ mỗi bước chỉ là phép AND.
- Tham lam "ăn nhiều nhất" thường sai trong bài phủ (cover) — luôn đối chiếu vét cạn.
- Test lưới **không vuông** để bắt lỗi đánh số bit.

## 15. Biến thể của bài

1. **LeetCode 2128 — Remove All Ones With Row and Column Flips (I):** **đảo** (flip) cả hàng/cột — bài toán tuyến tính mod 2: mọi hàng phải bằng hàng đầu hoặc phần bù của nó.
2. **Không bắt buộc ô chọn là 1** — số thao tác có thể giảm (lỗi 2 chính là biến thể này).
3. **LeetCode 1284 — Minimum Number of Flips to Convert Binary Matrix to Zero Matrix:** BFS trên bitmask, thao tác đảo ô + 4 hàng xóm.
4. **Trả về dãy thao tác cụ thể** (truy vết BFS) — bài tập 4.
5. **Chỉ xóa hàng** (hoặc chỉ cột): đáp án = số hàng có 1 — tầm thường; so sánh để thấy giá trị của việc xóa đồng thời.

In [21]:
def flips_possible_2128(grid):
    """Biến thể 1 (LeetCode 2128): có thể đưa về toàn 0 bằng đảo hàng/cột ⇔ mọi hàng bằng hàng 0 hoặc phần bù của nó."""
    first = grid[0]; inv = [1 - x for x in first]
    return all(row == first or row == inv for row in grid)

def flips_brute(grid):
    m, n = len(grid), len(grid[0])
    for rm in range(1 << m):
        for cm in range(1 << n):
            if all(grid[i][j] ^ (rm >> i & 1) ^ (cm >> j & 1) == 0 for i in range(m) for j in range(n)):
                return True
    return False

rnd_v = random.Random(15)
for _ in range(500):
    m, n = rnd_v.randint(1, 4), rnd_v.randint(1, 4)
    g = [[rnd_v.randint(0, 1) for _ in range(n)] for _ in range(m)]
    assert flips_possible_2128(g) == flips_brute(g)
print("2128: [[0,1,0],[1,0,1],[0,1,0]] ->", flips_possible_2128([[0, 1, 0], [1, 0, 1], [0, 1, 0]]), "| khớp vét cạn trên 500 lưới")

2128: [[0,1,0],[1,0,1],[0,1,0]] -> True | khớp vét cạn trên 500 lưới


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | thử mọi thứ tự | mũ | `O(K)` | kiểm thử lưới rất nhỏ |
| 2. DFS + memo | `f(mask)` | `O(R K)` | `O(R)` | ngắn gọn |
| 3. BFS | tầng | `O(R K)` dừng sớm | `O(R)` | **nộp bài** |
| 4. DP mọi tập con | thứ tự số tăng | `O(2^p K)` | `O(2^p)` | không đệ quy |
| 5. IDDFS + cận | sâu dần | mũ / nhanh thực tế | `O(R)` | khi không gian lớn hơn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().removeOnes([[1, 1], [1, 1]])`?

**Bài 2 (tính tay):** Với lưới `2 × 3` `[[1,0,1],[0,1,0]]`, liệt kê mask, mặt nạ xóa của ô `(1,1)`, và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `reachable_states(grid)` — số trạng thái (mask) đạt được từ lưới ban đầu (kể cả trạng thái đầu và `0`).

**Bài 4 (tự viết lại):** Viết lại BFS trả về **dãy ô** `[(i, j), …]` tối ưu; viết validator mô phỏng lại dãy thao tác (mỗi ô chọn phải đang là 1) và kiểm lưới cuối toàn 0.

**Bài 5 (mini challenge):** Biến thể "không bắt buộc ô chọn là 1" (mục 15.2): viết `remove_ones_any_cell(grid)` và tìm (bằng vét cạn ngẫu nhiên) lưới nhỏ nhất mà đáp án hai bài **khác nhau**.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def reachable_states(grid):
    # TODO: BFS/DFS từ start, đếm len(seen) (kể cả 0)
    pass

# Bài 4 — TODO
def remove_ones_path(grid):
    # TODO: BFS lưu parent[mask] = (mask_cũ, bit)
    pass

# Bài 5 — TODO
def remove_ones_any_cell(grid):
    # TODO: như memo nhưng duyệt mọi ô b trong range(m*n)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Chọn `(0,0)` xóa hàng 0 và cột 0.
- Mức 2: Còn `(1,1)`.
- Mức 3: Đáp án `2`.

**Bài 2**
- Mức 1: Bit `i·3 + j`: ô `(0,0)`→0, `(0,2)`→2, `(1,1)`→4 ⇒ mask `0b10101 = 21`.
- Mức 2: Mặt nạ xóa `(1,1)`: hàng 1 (bit 3,4,5) + cột 1 (bit 1,4) ⇒ `0b111010`.
- Mức 3: Chọn `(0,0)` xóa `(0,2)` cùng hàng, còn `(1,1)` ⇒ 2 thao tác; không có cách 1 thao tác.

**Bài 3**
- Mức 1: BFS không dừng sớm.
- Mức 2: `seen` chứa mọi trạng thái.
- Mức 3: So với số khóa trong cache của cách 2 (+1 cho trạng thái 0).

**Bài 4**
- Mức 1: `parent[nxt] = (mask, b)` khi lần đầu thấy `nxt`.
- Mức 2: Truy vết từ `0` về `start`.
- Mức 3: Validator: mô phỏng trên lưới thật.

**Bài 5**
- Mức 1: Đổi `bits(mask)` thành `range(m*n)`, nhưng **bỏ** các ô mà hàng + cột không chứa số 1 nào (thao tác vô ích, nếu không sẽ đệ quy vô hạn).
- Mức 2: Đáp án luôn `≤` bài gốc.
- Mức 3: Ví dụ 2 của đề là một lưới khác nhau (2 vs 1) — tìm lưới nhỏ hơn nữa nếu có.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = Solution().removeOnes([[1, 1], [1, 1]])
print("Bài 1:", r1); assert r1 == 2 == remove_ones_brute([[1, 1], [1, 1]])

# Bài 2
g2 = [[1, 0, 1], [0, 1, 0]]
print("Bài 2: mask =", to_mask(g2), bin(to_mask(g2)), "| xóa (1,1) =", bin(clear_masks(2, 3)[4]), "| đáp án", Solution().removeOnes(g2))
assert to_mask(g2) == 21 and clear_masks(2, 3)[4] == 0b111010 and Solution().removeOnes(g2) == 2

# Bài 3
def reachable_states_sol(grid):
    m, n = len(grid), len(grid[0]); clear = clear_masks(m, n); start = to_mask(grid)
    seen = {start}; stack = [start]
    while stack:
        mask = stack.pop()
        for b in bits(mask):
            nxt = mask & ~clear[b]
            if nxt not in seen:
                seen.add(nxt); stack.append(nxt)
    return len(seen)

def memo_cache_size(grid):
    m, n = len(grid), len(grid[0]); clear = clear_masks(m, n)
    @lru_cache(maxsize=None)
    def f(mask):
        if mask == 0: return 0
        return 1 + min(f(mask & ~clear[b]) for b in bits(mask))
    f(to_mask(grid)); return f.cache_info().currsize

rnd = random.Random(3)
for _ in range(300):
    m = rnd.randint(1, 4); n = rnd.randint(1, min(4, 15 // m))
    g = [[rnd.randint(0, 1) for _ in range(n)] for _ in range(m)]
    assert reachable_states_sol(g) == memo_cache_size(g)      # cache chứa đúng mọi trạng thái đạt được (kể cả 0)
print("Bài 3 OK; ví dụ 1 có", reachable_states_sol(examples[0][0]), "trạng thái đạt được")

Bài 1: 2
Bài 2: mask = 21 0b10101 | xóa (1,1) = 0b111010 | đáp án 2
Bài 3 OK; ví dụ 1 có 14 trạng thái đạt được


In [24]:
# Bài 4
def remove_ones_path_sol(grid):
    m, n = len(grid), len(grid[0]); clear = clear_masks(m, n); start = to_mask(grid)
    if start == 0:
        return []
    parent = {start: None}; q = deque([start])
    while q:
        mask = q.popleft()
        for b in bits(mask):
            nxt = mask & ~clear[b]
            if nxt not in parent:
                parent[nxt] = (mask, b); q.append(nxt)
        if 0 in parent:
            break
    path = []; cur = 0
    while parent[cur] is not None:
        pm, b = parent[cur]; path.append(divmod(b, n)); cur = pm
    return path[::-1]

def valid_path(grid, path):
    g = [row[:] for row in grid]; m, n = len(g), len(g[0])
    for i, j in path:
        if g[i][j] != 1:
            return False
        for k in range(n): g[i][k] = 0
        for k in range(m): g[k][j] = 0
    return not any(map(any, g))

# Bài 5
def remove_ones_any_cell_sol(grid):
    m, n = len(grid), len(grid[0]); clear = clear_masks(m, n)
    @lru_cache(maxsize=None)
    def f(mask):
        if mask == 0: return 0
        return 1 + min(f(mask & ~clear[b]) for b in range(m * n) if mask & clear[b])   # bỏ thao tác không xóa được gì
    return f(to_mask(grid))

rnd = random.Random(5)
smallest = None
for _ in range(1500):
    m = rnd.randint(1, 3); n = rnd.randint(1, 3)
    g = [[rnd.randint(0, 1) for _ in range(n)] for _ in range(m)]
    p = remove_ones_path_sol(g)
    assert valid_path(g, p) and len(p) == Solution().removeOnes(g)
    a, b = remove_ones_any_cell_sol(g), Solution().removeOnes(g)
    assert a <= b
    if a != b and (smallest is None or m * n < len(smallest) * len(smallest[0])):
        smallest = g
print("Bài 4 OK; đường đi ví dụ 1:", remove_ones_path_sol(examples[0][0]))
if smallest is None:
    smallest = examples[1][0]
print("Bài 5: lưới nhỏ nhất tìm được mà hai bài khác nhau:", smallest, "->", remove_ones_any_cell_sol(smallest), "vs", Solution().removeOnes(smallest))

Bài 4 OK; đường đi ví dụ 1: [(0, 0), (1, 1)]
Bài 5: lưới nhỏ nhất tìm được mà hai bài khác nhau: [[0, 1], [1, 0]] -> 1 vs 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template BFS trên bitmask:**

```python
start = mask(grid); clear[b] = mặt nạ hàng + cột của ô b
q, seen, steps = deque([start]), {start}, 0
while q:
    steps += 1
    for _ in range(len(q)):
        mask = q.popleft(); x = mask
        while x:
            low = x & -x; x ^= low
            nxt = mask & ~clear[low.bit_length() - 1]
            if nxt == 0: return steps
            if nxt not in seen: seen.add(nxt); q.append(nxt)
```

**Checklist:**
- [ ] Lưới toàn 0 trả `0` trước BFS?
- [ ] Đánh số bit `i · n + j` (`n` = số cột)?
- [ ] Chỉ chọn ô **đang là 1**?
- [ ] Mặt nạ xóa gồm **cả hàng và cột**?
- [ ] Có `seen` / ghi nhớ?
- [ ] Đã test lưới không vuông, `1 × 15`, `15 × 1`?
- [ ] Chữ ký đúng: `def removeOnes(self, grid: List[List[int]]) -> int`? (đối chiếu đề gốc nếu có Premium)